# Visual validation of rectangular grids

This notebook illustrates the conventions tested by PR #50:

- fields are shaped `(n_wavelengths, ny, nx)`;
- x is the horizontal, last tensor axis;
- y is the vertical, second-to-last tensor axis;
- tip shifts the PSF along x and tilt shifts it along y;
- chromatic field stops are shaped `(n_wavelengths, ny, nx)`.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux import (
    CircularAperture,
    Grid,
    MFTPropagator,
    PlaneWave,
    ShanonFieldStop,
    Spectrum,
    TipTilt,
)
from fiatlux.core.spectrum import Band

In [ ]:
# Deliberately rectangular pupil and focal grids.
nx_in, ny_in = 96, 64
nx_out, ny_out = 128, 80
diameter_x, diameter_y = 1.0, 0.8
wavelength = 1.0e-6
focal_length = 2.0

pupil_grid = Grid(
    nx=nx_in,
    ny=ny_in,
    dx=diameter_x / nx_in,
    dy=diameter_y / ny_in,
)
focal_grid = Grid(
    nx=nx_out,
    ny=ny_out,
    dx=wavelength * focal_length / (nx_in * pupil_grid.dx),
    dy=wavelength * focal_length / (ny_in * pupil_grid.dy),
)

spectrum = Spectrum(
    magnitude=0,
    band=Band(wavelength, 0.0, 3.68e8),
    samples=1,
)
source = PlaneWave(spectrum)
aperture = CircularAperture(pupil_grid, radius=0.35)
propagator = MFTPropagator(focal_length, focal_grid)

entrance_field = source.generate_field(pupil_grid)
pupil_field = aperture.apply(entrance_field)
focal_field = propagator.apply(pupil_field)
psf = focal_field.intensity()[0]

print('Entrance field:', tuple(entrance_field.complex_amplitude.shape))
print('Focal field:   ', tuple(focal_field.complex_amplitude.shape))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].imshow(
    pupil_field.intensity()[0].cpu(),
    origin='lower',
    extent=[pupil_grid.x.min(), pupil_grid.x.max(), pupil_grid.y.min(), pupil_grid.y.max()],
)
axes[0].set(title='Rectangular array, circular pupil', xlabel='x (m)', ylabel='y (m)')

axes[1].imshow((psf / psf.max()).cpu(), origin='lower', cmap='magma')
axes[1].set(title='Normalized focal-plane PSF', xlabel='x pixel', ylabel='y pixel')
plt.show()

## Tip and tilt orientation

The OPD ramps below correspond to exactly one Fourier bin. The expected peaks are one pixel to the right for tip and one pixel upward for tilt.

In [ ]:
def shifted_psf(tip=0.0, tilt=0.0):
    ramp = TipTilt(pupil_grid, tip=tip, tilt=tilt)
    ramped_field = ramp.apply(pupil_field)
    return propagator.apply(ramped_field).intensity()[0]

tip_one_bin = wavelength / (nx_in * pupil_grid.dx)
tilt_one_bin = wavelength / (ny_in * pupil_grid.dy)
images = [
    ('Reference', shifted_psf()),
    ('Tip: +1 x bin', shifted_psf(tip=tip_one_bin)),
    ('Tilt: +1 y bin', shifted_psf(tilt=tilt_one_bin)),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, (title, image) in zip(axes, images):
    peak_y, peak_x = torch.unravel_index(image.argmax(), image.shape)
    ax.imshow((image / image.max()).cpu(), origin='lower', cmap='magma')
    ax.scatter(peak_x.cpu(), peak_y.cpu(), marker='+', s=120, c='cyan')
    ax.set_title(f'{title}\npeak=(y={peak_y.item()}, x={peak_x.item()})')
    ax.set(xlabel='x pixel', ylabel='y pixel')
plt.show()

## Chromatic rectangular field stop

In [ ]:
polychromatic = Spectrum(
    magnitude=0,
    band=Band(wavelength, 0.4 * wavelength, 3.68e8),
    samples=3,
)
field_stop = ShanonFieldStop(focal_grid)
field_stop.build(polychromatic)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for index, (ax, wl) in enumerate(zip(axes, polychromatic.wavelengths)):
    ax.imshow(field_stop.transmission[index].real.cpu(), origin='lower', vmin=0, vmax=1)
    ax.set_title(f'λ = {wl.item() * 1e9:.0f} nm')
    ax.set(xlabel='x pixel', ylabel='y pixel')

print('Field-stop transmission:', tuple(field_stop.transmission.shape))
plt.show()